In [0]:
%pip install streamlit twilio qrcode geocoder pandas

In [0]:
import streamlit as st
import pandas as pd
import qrcode, io

st.title("RCCG Hope Centre – Transport Unit Dashboard")

tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "Bookings Overview",
    "Assign Drivers",
    "Driver Tracking",
    "Route Overview",
    "QR Check‑in"
])

with tab1:
    df = spark.table("rccg_hopecentre_transporthub.core.bookings").toPandas()
    st.subheader("All Sunday Bookings")
    st.dataframe(df)

with tab2:
    df = spark.table("rccg_hopecentre_transporthub.core.bookings").toPandas()
    booking_id = st.selectbox("Select Booking", df["booking_id"])
    driver = st.text_input("Assign Driver")

    if st.button("Save Assignment"):
        spark.sql(f"""
            UPDATE rccg_hopecentre_transporthub.core.bookings
            SET driver = '{driver}'
            WHERE booking_id = '{booking_id}'
        """)
        st.success("Driver assigned.")

with tab3:
    st.subheader("Live Driver Locations")
    loc_df = spark.table("rccg_hopecentre_transporthub.core.driver_locations").toPandas()
    if not loc_df.empty:
        st.map(loc_df[["latitude", "longitude"]])
    else:
        st.info("No driver locations yet.")

with tab4:
    st.subheader("Suggested Route Order")
    df = spark.table("rccg_hopecentre_transporthub.core.bookings").toPandas()
    route = df.sort_values(["location", "pickup_time"])
    st.dataframe(route[["location", "pickup_time", "driver"]])

with tab5:
    df = spark.table("rccg_hopecentre_transporthub.core.bookings").toPandas()
    booking_id = st.selectbox("Select booking for QR code", df["booking_id"])

    qr = qrcode.make(booking_id)
    buf = io.BytesIO()
    qr.save(buf)
    st.image(buf.getvalue(), caption="Show this QR to the driver")

    st.write("Driver check‑in is handled in a separate app.")
